<div style="
    text-align:center;
    margin-top:20px;
    border:2px solid grey;
    border-radius:18px;
    padding:30px 40px;
    max-width:1400px;
    margin-left:auto;
    margin-right:auto;
">

<h1 style="color:#003366; font-weight:750; margin-bottom:10px; letter-spacing:1px;">
PROYECTO FINAL
</h1>

<h2 style="color:#C9A227; font-weight:600; margin-top:0;">
🌍 Global Trends and Energy Challenges 🏭
</h2>

<p style="color:#555; font-style:italic; font-size:18px; margin-top:10px;">
A Data-Driven Analysis of the Energy Transition and Global Inequalities
</p>

<br>

<p style="font-size:16px; color:white; margin:0;">
<b>Francesco Eirado Enes</b>
</p>

<p style="color:#777; margin-top:2px;">
23 April 2026
</p>

<br>

<p style="color:#666; font-size:18px; max-width:950px; margin:auto;">
⚡ Exploring how energy consumption, ☢️ nuclear energy and 🌱 sustainability intersect across the globe.
</p>

</div>

<hr style="
    border: 0;
    height: 2px;
    background-color:#C9A227;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:#C9A227; font-size:28px;">
Prediction Global Countries vs Target 2030 Ember
</h2>
<h2 style="text-align:center; color:#777777; font-size:18px;">
I will build a linear‑regression model to predict each country’s energy trajectory up to 2030.<br>
These predictions will be compared against the official 2030 targets collected in the Ember dataset.<br>
The goal is to evaluate whether countries are on track, lagging behind, or exceeding their expected transition pathways.
</h2>

In [19]:
file_path = "00.ETL_Pipeline.ipynb"
with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if "pk.eyJ1" in line:
            print(f"¡ENCONTRADO! Línea {i+1}:")
            print(line.strip()[:100]) # Mostramos solo el principio por seguridad

In [20]:
# ============================
# Core
# ============================
import warnings
warnings.filterwarnings("ignore")
import logging

# ============================
# Standard Library
# ============================
import os
import json
import time
from datetime import datetime, date

# ============================
# Data Handling
# ============================
import pandas as pd
import numpy as np

# ============================
# Modeling & Statistics
# ============================
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_absolute_error
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
import statsmodels.api as sm

# ============================
# Visualization
# ============================
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================
# Mapping
# ============================
import branca.colormap as cm
import pydeck as pdk

# ============================
# Utilities
# ============================
import requests
import pycountry

# ============================
# Notebook Magic 
# ============================
%matplotlib inline

# ============================
# Prophet Logging Cleanup
# ============================
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

In [21]:
# Inspect df
def inspect_df(df):
    print("\n------ SHAPE ------")
    print(df.shape)

    print("\n------- NULLS ------")
    print(df.isna().sum().sort_values(ascending=False).head(150))

    print("\n------ DUPLICATES ------")
    print(df.duplicated().sum())

    print("\n------ DATA TYPES ------")
    print(df.dtypes.value_counts())

    print("\n------ UNIQUE VALUES ------")
    print(df.nunique().sort_values(ascending=False).head(150))

    print("\n------ SAMPLE ------")
    display(df.sample(n=min(5, len(df))))

    
# Save_df
def save_df(df, name, folder):
    path = fr"{folder}\{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")

In [22]:
# Load and Check df_prediction

path = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Prediction_df.csv"

df_prediction = pd.read_csv(path)

pd.set_option('display.max_columns', None)
inspect_df(df_prediction)


------ SHAPE ------
(16397, 28)

------- NULLS ------
2001        412
2000        401
2002        399
2003        341
2004        338
2005        265
2006        238
2007        221
2010        216
2011        203
2008        195
2009        193
2024        135
2012        130
2013        115
2014         97
2023         91
2016         87
2018         81
2017         81
2015         79
2019         75
2020         74
2021         72
2022         68
iso_code      0
variable      0
country       0
dtype: int64

------ DUPLICATES ------
0

------ DATA TYPES ------
float64    25
object      3
Name: count, dtype: int64

------ UNIQUE VALUES ------
2020        6212
2024        6199
2023        6184
2022        6181
2021        6064
2019        6045
2018        5930
2017        5832
2016        5784
2015        5783
2014        5731
2013        5665
2012        5609
2011        5521
2009        5457
2010        5440
2008        5350
2007        5273
2006        5198
2005        5111
2004   

,country,iso_code,variable,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
14051,Sweden,SWE,hydro_consumption,218.29,218.40,181.8,145.89,164.20,194.63,164.75,175.98,182.80,172.02,174.94,173.74,203.45,156.93,162.25,190.66,156.12,162.66,154.60,161.63,178.48,181.76,171.52,161.20,157.64
11589,Poland,POL,nuclear_share_elec,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3974,Djibouti,DJI,fossil_electricity,0.18,0.19,0.2,0.20,0.22,0.30,0.31,0.32,0.34,0.35,0.38,0.39,0.39,0.43,0.10,0.18,0.09,0.06,0.05,0.09,0.13,0.13,0.13,0.13,0.15
12061,Romania,ROU,wind_elec_per_capita,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.49,15.17,68.53,130.96,225.29,310.49,355.29,333.21,376.34,322.48,347.15,358.39,341.85,365.22,394.91,338.15
16130,Vietnam,VNM,other_renewables_capacity_gw,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


<hr style="
    border: 0;
    height: 1px;
    background-color:#C9A227;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:#C9A227; font-size:28px;"> 
Predict the model using only the Ember 2030 targets
</h2>

In [23]:
metrics_target = [
    "hydro_capacity_gw", "renewables_share_elec", "solar_capacity_gw",
    "wind_capacity_gw", "low_carbon_share_elec", "renewables_capacity_gw",
    "fossil_capacity_gw", "fossil_electricity", "fossil_share_elec",
    "hydro_electricity", "hydro_share_elec", "nuclear_capacity_gw",
    "nuclear_electricity", "nuclear_share_elec", "coal_capacity_gw",
    "coal_electricity", "coal_share_elec", "gas_capacity_gw",
    "gas_electricity", "gas_share_elec", "solar_electricity",
    "solar_share_elec", "wind_electricity", "wind_share_elec",
    "biofuel_capacity_gw", "biofuel_electricity", "biofuel_share_elec",
    "renewables_electricity", "renewables_share_energy"]

df_prediction = df_prediction[df_prediction["variable"].isin(metrics_target)]

<hr style="
    border: 0;
    height: 1px;
    background-color:GOLD;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:GOLD; font-size:28px;">
Preparing Owid_WB_Irena for the merge with Ember
</h2>

<h2 style="text-align:center; color:grey; font-size:18px;">

To prepare the <b>Owid_WB_Irena</b> dataset for merging with Ember,  
we standardize country identifiers, align the 2000–2024 timeline,  
and reshape all variables into a clean country‑year structure.

These transformations ensure full compatibility with Ember’s electricity data  
and allow the creation of a unified, consistent dataset ready for the 2030 prediction pipeline.

</h2>


In [24]:
# Disable warnings for NaN values during mathematical divisions
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ============================================================
# CONFIGURATION
# ============================================================

HISTORICAL_YEARS = [str(y) for y in range(2000, 2025)]
FUTURE_YEARS = [str(y) for y in range(2025, 2031)]

BASE_METRICS = [
    "hydro", "solar", "wind", "biofuel", 
    "nuclear", "coal", "gas"
]

# ============================================================
# BASE PREDICTIVE FUNCTION (Absolute Values Only)
# ============================================================

def predict_time_series(y_hist, years_hist, years_future, metric_name):
    """Esegue una regressione lineare con regole specifiche in base alla fonte."""
    mask = ~np.isnan(y_hist)
    if mask.sum() < 3: 
        return np.full(len(years_future), np.nan)
        
    y_clean = y_hist[mask]
    X_clean = years_hist[mask].reshape(-1, 1)
    X_future = years_future.reshape(-1, 1)
    
    # 1. WIND AND SOLAR RULES
    if "solar" in metric_name or "wind" in metric_name:
      
        recent_n = min(7, len(y_clean))
        X_fit = X_clean[-recent_n:]
        y_fit = y_clean[-recent_n:]
        
        model = LinearRegression()
        model.fit(X_fit, y_fit)
        pred = model.predict(X_future)
        
     
        pred = np.maximum(pred, 0)
        
    # 2. HIDRO RULES
    elif "hydro" in metric_name:
     
        recent_mean = np.nanmean(y_clean[-5:])
        pred = np.full(len(years_future), recent_mean)

    # 3. FOSSIL RULES
    elif "coal" in metric_name or "gas" in metric_name:
        model = LinearRegression()
        model.fit(X_clean, y_clean)
        pred = model.predict(X_future)
       
        pred = np.maximum(pred, 0)
        
    # 4. Default RULES
    else:
        model = LinearRegression()
        model.fit(X_clean, y_clean)
        pred = model.predict(X_future)
        
        pred = np.maximum(pred, 0)
        if "share" in metric_name:
            pred = np.minimum(pred, 100) # La share non supera mai il 100%
            
    return pred


# ============================================================
# PIPELINE 
# ============================================================

def generate_2030_predictions(df_storico):
    """
    df_storico: Pandas DataFrame containing the historical data.
    It must include: 'iso_code', 'country', 'variable', and the years '2000'...'2024'.
    """

    
    years_hist_int = np.array([int(y) for y in HISTORICAL_YEARS])
    years_future_int = np.array([int(y) for y in FUTURE_YEARS])
    
    results = []
    
  
    countries = df_storico[['iso_code', 'country']].drop_duplicates()
    
    for _, c_row in countries.iterrows():
        iso = c_row['iso_code']
        country_name = c_row['country']
        
      
        df_country = df_storico[df_storico['iso_code'] == iso]
        
        future_data = {} 
        storico_data = {}
        
       # -------------------------------------------------------------
       # PHASE 1: PREDICT THE BASE VARIABLES
       # -------------------------------------------------------------

        variabili_da_predire = [f"{b}_capacity_gw" for b in BASE_METRICS] + \
                               [f"{b}_electricity" for b in BASE_METRICS] + \
                               ["renewables_share_energy"]
                               
        for var in variabili_da_predire:
            row = df_country[df_country['variable'] == var] 
            
            # Check columns
            if row.empty and 'energy_metric' in df_country.columns:
                row = df_country[df_country['energy_metric'] == var]
            
            if row.empty:
                storico_data[var] = np.full(len(HISTORICAL_YEARS), np.nan)
                future_data[var] = np.full(len(FUTURE_YEARS), np.nan)
                continue
                
            row = row.iloc[0]
            y_hist = pd.to_numeric(row[HISTORICAL_YEARS], errors='coerce').values.astype(float)
            storico_data[var] = y_hist
            future_data[var] = predict_time_series(y_hist, years_hist_int, years_future_int, var)
            
        # -------------------------------------------------------------
        # PHASE 2: COMPUTATION OF AGGREGATES (Renewables and Fossils)
        # -------------------------------------------------------------

        def safe_sum(keys, data_dict, length):
            arrays = [data_dict[k] for k in keys if k in data_dict]
            if not arrays: return np.full(length, np.nan)
            return np.nansum(np.array(arrays), axis=0)

        for data_dict, dict_length in [(storico_data, len(HISTORICAL_YEARS)), (future_data, len(FUTURE_YEARS))]:
            
            data_dict["renewables_capacity_gw"] = safe_sum([f"{s}_capacity_gw" for s in ["hydro", "solar", "wind", "biofuel"]], data_dict, dict_length)
            data_dict["fossil_capacity_gw"] = safe_sum([f"{s}_capacity_gw" for s in ["coal", "gas"]], data_dict, dict_length)
            
            data_dict["renewables_electricity"] = safe_sum([f"{s}_electricity" for s in ["hydro", "solar", "wind", "biofuel"]], data_dict, dict_length)
            data_dict["fossil_electricity"] = safe_sum([f"{s}_electricity" for s in ["coal", "gas"]], data_dict, dict_length)
            data_dict["low_carbon_electricity"] = safe_sum(["renewables_electricity", "nuclear_electricity"], data_dict, dict_length)
            
            total_elec = safe_sum([f"{s}_electricity" for s in BASE_METRICS], data_dict, dict_length)
            
            # -------------------------------------------------------------
            # PHASE 3: CALCULATION SHARE ELEC AND FALLBACK ENERGY
            # -------------------------------------------------------------
            safe_total = np.where(total_elec == 0, np.nan, total_elec)
            
            for source in BASE_METRICS:
                data_dict[f"{source}_share_elec"] = (data_dict[f"{source}_electricity"] / safe_total) * 100
                
            data_dict["renewables_share_elec"] = (data_dict["renewables_electricity"] / safe_total) * 100
            data_dict["fossil_share_elec"] = (data_dict["fossil_electricity"] / safe_total) * 100
            data_dict["low_carbon_share_elec"] = (data_dict["low_carbon_electricity"] / safe_total) * 100

            # FALLBACK LOGIC 
            if "renewables_share_energy" in data_dict:
                if np.isnan(data_dict["renewables_share_energy"]).all():
                    data_dict["renewables_share_energy"] = data_dict["renewables_share_elec"]

        # -------------------------------------------------------------
        # PHASE 4: SAVE
        # -------------------------------------------------------------
        target_metrics = [
            "hydro_capacity_gw", "solar_capacity_gw", "wind_capacity_gw", "biofuel_capacity_gw", 
            "nuclear_capacity_gw", "coal_capacity_gw", "gas_capacity_gw", "renewables_capacity_gw", "fossil_capacity_gw",
            "hydro_share_elec", "solar_share_elec", "wind_share_elec", "biofuel_share_elec", "nuclear_share_elec", 
            "coal_share_elec", "gas_share_elec", "renewables_share_elec", "low_carbon_share_elec", "fossil_share_elec", 
            "renewables_share_energy",
            "hydro_electricity", "solar_electricity", "wind_electricity", "biofuel_electricity", 
            "nuclear_electricity", "coal_electricity", "gas_electricity", "renewables_electricity", "fossil_electricity"
        ]
        
        for metric in target_metrics:
            row_out = {
                "iso_code": iso,
                "country": country_name,
                "variable": metric
            }
            for i, year in enumerate(HISTORICAL_YEARS):
                row_out[year] = storico_data[metric][i]
            for i, year in enumerate(FUTURE_YEARS):
                row_out[year] = future_data[metric][i]
                
            results.append(row_out)

    return pd.DataFrame(results).round(3)

print("Model ready to use!")

Model ready to use!


In [25]:
# Run 
df_final = generate_2030_predictions(df_prediction)

df_final_analisis = df_final.copy()


# Change name column
df_final_analisis = df_final_analisis.rename(columns={"variable": "energy_metric"})

In [26]:
df_final_analisis.tail(10)

,iso_code,country,energy_metric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
6283,ZWE,Zimbabwe,renewables_share_energy,46.098,37.611,44.457,60.82,56.701,52.57,68.136,71.863,76.184,75.928,68.097,57.941,60.549,54.114,55.285,52.61,45.161,54.993,55.97,51.957,58.921,70.712,67.338,67.509,69.987,64.720,65.041,65.365,65.690,66.017,66.346
6284,ZWE,Zimbabwe,hydro_electricity,3.190,2.960,3.810,5.34,5.500,4.91,5.310,5.380,5.710,5.460,5.800,5.200,5.390,5.000,5.430,4.99,2.980,3.970,5.05,4.170,3.810,5.930,5.880,5.460,5.040,5.224,5.224,5.224,5.224,5.224,5.224
6285,ZWE,Zimbabwe,solar_electricity,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.01,0.010,0.010,0.01,0.020,0.020,0.020,0.030,0.030,0.030,0.036,0.039,0.042,0.045,0.049,0.052
6286,ZWE,Zimbabwe,wind_electricity,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6287,ZWE,Zimbabwe,biofuel_electricity,0.000,0.000,0.000,0.00,0.000,0.00,0.100,0.060,0.080,0.060,0.070,0.090,0.120,0.130,0.010,0.04,0.230,0.150,0.19,0.190,0.100,0.110,0.110,0.120,0.130,0.169,0.176,0.182,0.189,0.195,0.202
6288,ZWE,Zimbabwe,nuclear_electricity,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6289,ZWE,Zimbabwe,coal_electricity,3.730,4.910,4.760,3.44,4.200,4.43,2.530,2.130,1.810,1.750,2.750,3.840,3.590,4.350,4.400,4.54,3.910,3.380,4.13,4.050,2.740,2.510,2.920,2.700,2.230,2.959,2.923,2.887,2.851,2.815,2.779
6290,ZWE,Zimbabwe,gas_electricity,0.000,0.000,0.000,0.00,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6291,ZWE,Zimbabwe,renewables_electricity,3.190,2.960,3.810,5.34,5.500,4.91,5.410,5.440,5.790,5.520,5.870,5.290,5.510,5.130,5.440,5.04,3.220,4.130,5.25,4.380,3.930,6.060,6.020,5.610,5.200,5.429,5.439,5.448,5.458,5.468,5.478
6292,ZWE,Zimbabwe,fossil_electricity,3.730,4.910,4.760,3.44,4.200,4.43,2.530,2.130,1.810,1.750,2.750,3.840,3.590,4.350,4.400,4.54,3.910,3.380,4.13,4.050,2.740,2.510,2.920,2.700,2.230,2.959,2.923,2.887,2.851,2.815,2.779


In [27]:
# Save df_final_analisis

path = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global"
save_df(df_final_analisis,"Df_final_analisis", path)

<hr style="
    border: 0;
    height: 2px;
    background-color:#C9A227;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:#C9A227; font-size:32px;">
ANALYSIS and CHARTS
</h2>

In [28]:
# Load df_target_2030
path = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Df_target_2030.csv"
df_target_2030 = pd.read_csv(path)


pd.set_option('display.max_rows', None)
df_target_2030.tail(10).round(20)

,iso_code,country,energy_metric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2030_target
966,UZB,Uzbekistan,nuclear_electricity,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,18.000
967,UZB,Uzbekistan,coal_electricity,1.910,1.960,2.020,1.990,2.030,1.95,2.020,2.000,2.050,2.050,2.120,2.150,2.160,2.230,2.300,1.940,1.990,2.090,2.350,3.820,5.200,4.960,5.640,5.700,6.320,4.815,4.968,5.120,5.272,5.424,5.576,10.100
968,UZB,Uzbekistan,gas_electricity,36.710,36.890,36.710,34.400,37.670,33.63,39.760,40.300,42.200,40.530,40.900,44.400,43.800,46.400,47.180,48.500,49.710,50.650,54.430,52.560,55.840,60.060,61.030,62.000,65.520,62.293,63.510,64.728,65.945,67.162,68.379,60.600
969,VNM,Vietnam,hydro_capacity_gw,3.270,3.810,4.120,4.170,4.210,4.27,4.510,4.910,5.910,7.180,8.650,10.080,13.550,14.720,15.750,16.630,17.850,17.810,17.990,20.330,20.820,21.850,22.540,22.920,23.710,22.368,22.368,22.368,22.368,22.368,22.368,29.346
970,VNM,Vietnam,solar_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.010,0.010,0.010,0.010,0.100,4.990,16.660,16.660,16.700,18.590,18.670,25.046,28.008,30.971,33.933,36.896,39.858,22.580
971,VNM,Vietnam,wind_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.010,0.030,0.030,0.030,0.050,0.050,0.140,0.160,0.200,0.240,0.370,0.520,4.120,5.070,5.890,6.180,7.971,9.165,10.358,11.551,12.744,13.938,27.880
972,VNM,Vietnam,biofuel_capacity_gw,0.120,0.120,0.120,0.120,0.120,0.13,0.130,0.130,0.130,0.130,0.130,0.130,0.130,0.130,0.200,0.220,0.260,0.270,0.380,0.390,0.390,0.370,0.390,0.410,0.460,0.414,0.429,0.443,0.458,0.473,0.488,2.270
973,VNM,Vietnam,coal_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9.760,12.900,14.450,16.590,17.810,19.740,21.550,24.670,25.310,26.760,26.760,25.722,27.035,28.348,29.661,30.975,32.288,30.127
974,VNM,Vietnam,gas_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,7.250,7.730,7.500,7.500,7.500,7.340,7.340,7.150,7.150,7.150,7.530,8.897,9.332,9.767,10.201,10.636,11.071,37.330
975,VNM,Vietnam,renewables_share_elec,66.016,70.445,57.943,49.621,40.027,33.22,34.991,36.618,37.518,38.106,32.089,40.871,46.432,46.844,44.326,34.994,36.109,44.462,38.849,30.684,35.083,42.908,49.597,43.206,42.369,43.899,44.176,44.429,44.662,44.876,45.074,47.000


In [29]:
def classify_gap(pred, target, threshold=0.10):
    if pd.isna(target):
        return "No target"
    if pd.isna(pred):
        return "No data"
    if target == 0:
        if pred == 0:
            return "In linea"
        else:
            return "Sopra target"
    
    gap = (pred - target) / target
    
    if abs(gap) <= threshold:
        return "In linea"
    elif gap > threshold:
        return "Sopra target"
    else:
        return "Sotto target"

df_gap = df_target_2030.copy()

df_gap["gap_%"] = np.where(
    df_gap["2030_target"] == 0,
    np.nan,
    ((df_gap["2030"] - df_gap["2030_target"]) / df_gap["2030_target"]) * 100
)

df_gap["giudizio"] = df_gap.apply(lambda r: classify_gap(r["2030"], r["2030_target"]), axis=1)

df_gap = df_gap.sort_values(["iso_code", "energy_metric"]).reset_index(drop=True)

df_gap.head(30)

,iso_code,country,energy_metric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2030_target,gap_%,giudizio
0,ALB,Albania,hydro_capacity_gw,1.450,1.450,1.450,1.450,1.450,1.450,1.450,1.460,1.460,1.460,1.480,1.510,1.630,1.780,1.720,1.800,1.910,2.050,2.100,2.160,2.390,2.510,2.490,2.520,2.520,2.486,2.486,2.486,2.486,2.486,2.486,2.510,-0.956175,In linea
1,ALB,Albania,renewables_share_elec,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,0.000000,In linea
2,ALB,Albania,solar_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.010,0.020,0.020,0.100,0.210,0.310,0.297,0.347,0.398,0.448,0.499,0.549,0.490,12.040816,Sopra target
3,ALB,Albania,wind_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.300,-100.000000,Sotto target
4,ARE,United Arab Emirates,low_carbon_share_elec,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.012,0.021,0.020,0.019,0.082,0.266,0.236,0.247,0.557,0.882,2.644,4.960,11.271,17.919,29.314,31.733,17.957,18.956,19.879,20.735,21.531,22.273,32.000,-30.396875,Sotto target
5,ARE,United Arab Emirates,renewables_capacity_gw,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.010,0.010,0.010,0.010,0.130,0.130,0.130,0.140,0.350,0.600,1.940,2.330,3.000,3.610,6.070,6.820,7.397,8.378,9.358,10.339,11.319,12.300,9.200,33.695652,Sopra target
6,ARG,Argentina,fossil_capacity_gw,12.830,14.120,14.260,14.260,13.170,12.790,11.950,10.810,11.670,13.530,12.520,12.400,13.360,17.710,17.960,16.230,16.540,18.780,20.560,20.690,21.440,21.490,21.360,21.900,21.860,22.035,22.486,22.938,23.389,23.840,24.292,31.585,-23.090074,Sotto target
7,ARG,Argentina,fossil_electricity,50.660,44.620,41.520,48.690,56.890,57.880,58.390,62.520,67.350,63.450,66.820,70.770,78.870,77.080,78.700,81.500,86.210,93.220,97.380,92.480,90.360,92.310,78.940,78.150,83.260,97.760,99.747,101.735,103.722,105.710,107.697,97.681,10.253785,Sopra target
8,ARG,Argentina,fossil_share_elec,59.017,50.321,49.594,53.494,59.162,57.903,55.477,61.222,62.832,59.034,61.415,64.465,67.839,65.289,66.274,66.276,68.431,70.095,69.821,68.150,65.961,66.163,61.346,56.320,56.969,61.385,60.639,59.939,59.280,58.660,58.075,43.020,34.995351,Sopra target
9,ARG,Argentina,hydro_capacity_gw,8.610,8.640,8.790,8.810,8.930,8.950,8.950,9.190,9.190,9.570,9.570,10.100,10.140,10.140,10.140,10.140,10.200,10.270,10.320,10.340,10.380,10.380,10.390,10.390,9.200,10.148,10.148,10.148,10.148,10.148,10.148,13.215,-23.208475,Sotto target


In [30]:
elec = df_gap[df_gap["energy_metric"].str.contains("electricity")].head(20)

elec

,iso_code,country,energy_metric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2030_target,gap_%,giudizio
7,ARG,Argentina,fossil_electricity,50.66,44.62,41.52,48.69,56.89,57.88,58.39,62.52,67.35,63.45,66.82,70.77,78.87,77.08,78.70,81.50,86.21,93.22,97.38,92.48,90.36,92.31,78.94,78.15,83.26,97.760,99.747,101.735,103.722,105.710,107.697,97.681,10.253785,Sopra target
10,ARG,Argentina,hydro_electricity,28.58,36.58,35.78,33.80,30.46,34.22,38.02,31.22,31.20,34.62,33.36,31.19,29.16,32.77,32.42,32.34,29.75,31.59,32.28,27.50,23.67,19.65,22.77,31.49,29.67,25.450,25.450,25.450,25.450,25.450,25.450,49.286,-48.362618,Sotto target
13,ARG,Argentina,nuclear_electricity,6.18,7.06,5.82,7.57,7.87,6.87,7.72,7.25,7.39,8.18,7.21,6.40,6.40,6.21,5.51,7.04,8.28,6.09,6.87,8.44,10.01,10.17,7.47,8.96,10.45,8.755,8.852,8.948,9.045,9.142,9.239,21.785,-57.590085,Sotto target
16,AUS,Australia,coal_electricity,180.26,179.88,171.86,173.61,178.96,182.98,185.76,185.49,184.24,182.02,176.02,171.98,165.38,155.45,155.23,162.20,162.36,159.17,156.56,149.84,142.93,137.40,129.21,125.61,127.33,133.103,130.782,128.462,126.141,123.821,121.500,40.700,198.525799,Sopra target
19,AUS,Australia,gas_electricity,16.76,24.50,30.55,30.15,27.36,23.26,27.29,33.40,36.31,41.12,46.79,48.78,49.81,52.72,53.66,49.71,48.11,55.09,51.46,55.71,53.05,47.63,50.82,46.67,49.10,59.680,61.041,62.401,63.762,65.122,66.483,5.500,1108.781818,Sopra target
22,AUS,Australia,hydro_electricity,16.47,16.12,16.09,16.21,15.69,15.47,14.95,13.05,11.82,12.64,13.75,19.57,17.04,19.09,14.48,14.09,17.63,13.50,17.31,14.02,14.42,15.94,16.66,15.08,12.85,14.990,14.990,14.990,14.990,14.990,14.990,15.300,-2.026144,In linea
25,AUS,Australia,solar_electricity,0.05,0.05,0.06,0.06,0.07,0.08,0.10,0.11,0.14,0.29,0.98,2.04,2.41,3.85,4.95,6.20,7.44,8.92,12.33,18.30,23.84,31.19,37.54,44.99,50.21,57.017,63.471,69.926,76.380,82.834,89.289,77.000,15.959740,Sopra target
28,AUS,Australia,wind_electricity,0.13,0.29,0.53,0.70,0.80,1.30,2.16,2.85,3.46,4.44,4.99,6.43,7.72,9.26,9.78,11.83,13.04,13.21,16.26,19.47,22.61,26.80,30.05,31.87,32.74,37.354,40.271,43.189,46.106,49.023,51.940,106.500,-51.230047,Sotto target
31,AUT,Austria,biofuel_electricity,1.53,1.67,1.52,1.65,1.91,2.43,3.23,4.06,4.16,4.28,4.48,4.56,4.68,4.70,4.55,4.65,4.78,4.92,4.93,4.66,4.59,4.48,4.75,4.56,4.67,5.683,5.823,5.964,6.104,6.245,6.385,6.000,6.416667,In linea
33,AUT,Austria,fossil_electricity,13.57,15.64,15.92,19.60,18.86,20.20,18.04,16.50,16.71,16.10,19.27,17.87,14.10,10.86,8.26,10.61,10.53,12.67,11.73,12.82,10.51,10.62,10.85,7.50,7.54,9.185,9.127,9.069,9.012,8.954,8.896,6.000,48.266667,Sopra target


In [31]:
# =====================================================================
#  SMART CLASSIFICATION & SCORE COMPUTATION
# =====================================================================

def classify_smart_gap(pred, target, metric, threshold=0.10):
    if pd.isna(target): return "No target"
    if pd.isna(pred): return "No data"
    
    # Check if we are evaluating a fossil fuel metric
    is_fossil = any(word in str(metric).lower() for word in ["fossil", "coal", "gas", "oil"])
    
    # Handle the scenario where the target is absolute zero
    if target == 0:
        if pred == 0: return "On Track"
        return "Failing" if is_fossil else "Exceeding Goal"
            
    gap = (pred - target) / target
    
    # Inside the 10% tolerance margin
    if abs(gap) <= threshold: return "On Track"
        
    if not is_fossil:
        # Renewables: producing more is a success
        return "Exceeding Goal" if gap > threshold else "Failing"
    else:
        # Fossils: producing more is a failure
        return "Exceeding Goal" if gap < -threshold else "Failing"

# Apply the smart classification
df_gap["judgement"] = df_gap.apply(lambda r: classify_smart_gap(r["2030"], r["2030_target"], r["energy_metric"]), axis=1)

def calculate_smart_score(row):
    if row["judgement"] in ["No target", "No data"]: return np.nan
    if row["judgement"] in ["On Track", "Exceeding Goal"]: return 100.0
        
    is_fossil = any(word in str(row["energy_metric"]).lower() for word in ["fossil", "coal", "gas", "oil"])
    
    # If failing, deduct the gap from 100
    if not is_fossil:
        score = 100 + row["gap_%"] # Failing renewables: gap is negative (e.g. -25%). Score = 75
    else:
        score = 100 - row["gap_%"] # Failing fossils: gap is positive (e.g. +30%). Score = 70
        
    return max(0, min(100, score))

# Apply the score for the Radar Chart
df_gap["achievement_score"] = df_gap.apply(calculate_smart_score, axis=1)

# =====================================================================
# 2. FILTERING & CATEGORY MAPPING
# =====================================================================
df_valid = df_gap[~df_gap["judgement"].isin(["No target", "No data"])].copy()

cat_mapping = {
    "solar": "Solar", "wind": "Wind", "hydro": "Hydro",
    "biofuel": "Biofuel", "nuclear": "Nuclear", "coal": "Coal",
    "gas": "Natural Gas", "oil": "Oil", "fossil": "Total Fossil",
    "renewables": "Total Renewables", "low_carbon": "Low Carbon"
}

def get_cat(metric):
    for key, name in cat_mapping.items():
        if key in str(metric).lower(): return name
    return "Other"

df_valid["source_cat"] = df_valid["energy_metric"].apply(get_cat)

# Unified colors
color_map = {"Exceeding Goal": "#2ca02c", "On Track": "#f1c40f", "Failing": "#d62728"}
statuses = ["Exceeding Goal", "On Track", "Failing"]

all_cats = ["All Categories"] + sorted(df_valid["source_cat"].unique().tolist())



In [32]:
# =====================================================================
# CHART 1: DONUT CHART (GLOBAL SUCCESS RATE) - UPDATED ORDER
# =====================================================================
fig_pie = go.Figure()

# Count totals for the default view (All Categories)
df_pie_all = df_valid["judgement"].value_counts().reset_index()
df_pie_all.columns = ["judgement", "count"]


target_order = ["Exceeding Goal", "On Track", "Failing"]
df_pie_all["judgement"] = pd.Categorical(df_pie_all["judgement"], categories=target_order, ordered=True)
df_pie_all = df_pie_all.sort_values("judgement")

pie_colors = [color_map[s] for s in df_pie_all["judgement"]]

fig_pie.add_trace(go.Pie(
    labels=df_pie_all["judgement"],
    values=df_pie_all["count"],
    hole=0.4, 
    marker=dict(colors=pie_colors, line=dict(color='#cccccc', width=1)),
    textinfo='percent+label',
    hoverinfo='label+value',
    sort=False,             
    direction='clockwise',  
    rotation=90             
))

btn_pie = []
for cat in all_cats:
    if cat == "All Categories":
        df_c = df_valid
    else:
        df_c = df_valid[df_valid["source_cat"] == cat]
        
    df_pie_cat = df_c["judgement"].value_counts().reset_index()
    df_pie_cat.columns = ["judgement", "count"]
    
    
    df_pie_cat["judgement"] = pd.Categorical(df_pie_cat["judgement"], categories=target_order, ordered=True)
    df_pie_cat = df_pie_cat.sort_values("judgement")
    c_colors = [color_map[s] for s in df_pie_cat["judgement"]]
    
    btn_pie.append(dict(
        label=cat,
        method="update",
        args=[
            {"labels": [df_pie_cat["judgement"].tolist()],
             "values": [df_pie_cat["count"].tolist()],
             "marker": [{"colors": c_colors, "line": {"color": '#cccccc', "width": 1}}],
             "direction": "clockwise",
             "rotation": 90},
            {"title.text": f"Global Transition Status - {cat}"}
        ]
    ))

fig_pie.update_layout(
    title=dict(
        text="Global Transition Status - All Categories",
        x=0.5,
        xanchor="center",
        font=dict(size=28, color="#383838")
    ),
    template="plotly_white",
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7",
    font=dict(color="#383838", size=14),
    
    showlegend=True,
    legend=dict(
        orientation="v", 
        yanchor="top", y=1, 
        xanchor="left", x=1.02,
        font=dict(size=14, color="#383838")
    ),
    
    updatemenus=[dict(
        active=0,
        buttons=btn_pie,
        x=0.12, xanchor="left",
        y=1.15, yanchor="top",
        bgcolor="white",
        bordercolor="#4c4c4c",
        borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Filter:</b>",
            x=0.11, y=1.13,
            xref="paper", yref="paper",
            showarrow=False,
            xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120)
)

fig_pie.show()

# Save
fig_pie.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\19_prediction_donut.html")


In [33]:
# =====================================================================
# CHART 2: STACKED BAR CHART WITH CATEGORY FILTER - UPDATED ORDER
# =====================================================================

fig_bar = go.Figure()
df_cnt_all = df_valid.groupby(["energy_metric", "judgement"]).size().reset_index(name="count")

# Order stack
target_order = ["Failing", "On Track", "Exceeding Goal"]

for s in target_order:
    df_s = df_cnt_all[df_cnt_all["judgement"] == s]
    fig_bar.add_trace(go.Bar(
        x=df_s["energy_metric"], 
        y=df_s["count"], 
        name=s, 
        marker_color=color_map.get(s, "#000")
    ))

btn_bar = []
for cat in all_cats:
    df_c = df_valid if cat == "All Categories" else df_valid[df_valid["source_cat"] == cat]
    df_cnt = df_c.groupby(["energy_metric", "judgement"]).size().reset_index(name="count")
    
    x_arr, y_arr = [], []
    for s in target_order: 
        d = df_cnt[df_cnt["judgement"] == s]
        x_arr.append(d["energy_metric"].tolist())
        y_arr.append(d["count"].tolist())
        
    btn_bar.append(dict(
        label=cat, method="update",
        args=[{"x": x_arr, "y": y_arr}, {"title.text": f"2030 Scorecard: Countries per Status ({cat})"}]
    ))

fig_bar.update_layout(
    barmode="stack",
    title=dict(
        text="2030 Scorecard: Countries per Status (All Categories)",
        x=0.5,
        xanchor="center",
        font=dict(size=28, color="#383838")
    ),
    template="plotly_white", 
    plot_bgcolor="#FFF8E7", 
    paper_bgcolor="#FFF8E7", 
    font=dict(color="#383838", size=14),
    
    legend=dict(
        orientation="v", 
        yanchor="top", y=1, 
        xanchor="left", x=1.02,
        font=dict(size=14, color="#383838"),
        traceorder="reversed" 
    ),
    
    updatemenus=[dict(
        active=0, buttons=btn_bar, 
        x=0.12, xanchor="left",
        y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#4c4c4c", borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Filter:</b>",
            x=0.11, y=1.13,
            xref="paper", yref="paper",
            showarrow=False, xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120)
)


# Run
fig_bar.show()
# To save it:
fig_bar.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\20_prediction_bar.html")

In [34]:
# =====================================================================
# 4. CHART 3: BOX PLOT WITH CATEGORY FILTER
# =====================================================================

df_box_valid = df_valid[(df_valid["gap_%"] >= -100) & (df_valid["gap_%"] <= 300)].copy()
fig_box = go.Figure()

target_order = ["Exceeding Goal", "On Track", "Failing"]

for s in target_order:
    df_s_box = df_box_valid[df_box_valid["judgement"] == s]
    fig_box.add_trace(go.Box(
        x=df_s_box["energy_metric"], y=df_s_box["gap_%"], name=s, marker_color=color_map.get(s, "#000"),
        boxpoints="all", jitter=0.5, pointpos=-1.8, text=df_s_box["iso_code"], hoverinfo="text+y"
    ))

btn_box = []
for cat in all_cats:
    df_c = df_box_valid if cat == "All Categories" else df_box_valid[df_box_valid["source_cat"] == cat]
        
    x_b, y_b, t_b = [], [], []
    for s in target_order:
        d = df_c[df_c["judgement"] == s]
        x_b.append(d["energy_metric"].tolist())
        y_b.append(d["gap_%"].tolist())
        t_b.append(d["iso_code"].tolist())
        
    btn_box.append(dict(
        label=cat, method="update",
        args=[{"x": x_b, "y": y_b, "text": t_b}, {"title.text": f"2030 Target Gap Depth (%) - {cat}"}]
    ))

fig_box.update_layout(
    title=dict(
        text="2030 Target Gap Depth (%) - All Categories",
        x=0.5,
        xanchor="center",
        font=dict(size=28, color="#383838")
    ),
    template="plotly_white",
    plot_bgcolor="#FFF8E7",
    paper_bgcolor="#FFF8E7", 
    font=dict(color="#383838", size=14),
    yaxis=dict(gridcolor="rgba(0,0,0,0.1)"),
    
    legend=dict(
        orientation="v", 
        yanchor="top", y=1, 
        xanchor="left", x=1.02,
        font=dict(size=14, color="#383838")
    ),
    
    updatemenus=[dict(
        active=0, buttons=btn_box, 
        x=0.12, xanchor="left",
        y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#4c4c4c", borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Filter:</b>",
            x=0.11, y=1.13,
            xref="paper", yref="paper",
            showarrow=False, xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120)
)
fig_box.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Perfect Target (0%)")

# Run
fig_box.show()
# Save
fig_box.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\21_prediction_box.html")

In [35]:
# =====================================================================
# 5. CHART 3: RADAR CHART FILTERABLE BY COUNTRY - FILTER TOP-RIGHT
# =====================================================================

df_radar = df_valid.dropna(subset=["achievement_score"]).copy()
countries = sorted(df_radar["country"].unique().tolist())

fig_radar = go.Figure()
buttons_radar = []

for i, country in enumerate(countries):
    df_p = df_radar[df_radar["country"] == country].drop_duplicates(subset=["energy_metric"])
    
    r_val, th_val = df_p["achievement_score"].tolist(), df_p["energy_metric"].tolist()
    
    if len(r_val) > 2:
        r_val.append(r_val[0])
        th_val.append(th_val[0])
        
    fig_radar.add_trace(go.Scatterpolar(
        r=r_val, theta=th_val, fill='toself', name=country, visible=(i == 0),
        marker=dict(color='#4361ee', size=8), fillcolor='rgba(67, 97, 238, 0.4)',
        line=dict(color='#4361ee', width=2), hoverinfo="r+theta"
    ))
    
    vis = [False] * len(countries)
    vis[i] = True
    
    buttons_radar.append(dict(
        label=country, method="update",
        args=[{"visible": vis}, {"title.text": f"Target Achievement Profile: {country}"}]
    ))

fig_radar.update_layout(
    title=dict(
        text=f"Target Achievement Profile: {countries[0] if countries else ''}",
        x=0.5,
        xanchor="center",
        font=dict(size=28, color="#383838") 
    ),
    template="plotly_white", 
    plot_bgcolor="#FFF8E7", 
    paper_bgcolor="#FFF8E7", 
    font=dict(color="#383838", size=14),
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 100], gridcolor="rgba(0,0,0,0.1)"),
        angularaxis=dict(gridcolor="rgba(0,0,0,0.1)")
    ),
    showlegend=False,
    
    updatemenus=[dict(
        active=0, 
        buttons=buttons_radar, 
        x=1.0, xanchor="right",    
        y=1.15, yanchor="top", 
        bgcolor="white",
        bordercolor="#4c4c4c",
        borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Country:</b>", 
            x=0.88, y=1.13,      
            xref="paper", yref="paper",
            showarrow=False, 
            xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120)
)


# Show
fig_radar.show()

# To save it:
fig_radar.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\22_prediction_radar.html")

In [36]:
# --- GLOBAL CONFIGURATION ---
df = df_target_2030
title = "Global Electricity Transition"
filename = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\27_global_target_area.html"

low_carbon_metrics = [
    "solar_electricity", "wind_electricity", "hydro_electricity",
    "nuclear_electricity", "biofuel_electricity", "other_renewable_electricity"
]

fossil_metrics = ["coal_electricity", "gas_electricity"]   

color_map = {
    "solar_electricity": "#f1c40f",
    "wind_electricity": "#2ca02c",
    "hydro_electricity": "#1f77b4",
    "nuclear_electricity": "#9467bd",
    "biofuel_electricity": "#8c564b",
    "other_renewable_electricity": "#17becf",
    "coal_electricity": "#4c4c4c",
    "gas_electricity": "#C39BD3"
}


# 1. Data Preparation
year_cols = [c for c in df.columns if str(c).isdigit()]
df_melted = df.melt(id_vars=["energy_metric"], value_vars=year_cols,
                    var_name="year", value_name="value")

df_melted["year"] = df_melted["year"].astype(int)

df_yearly = (
    df_melted.groupby(["year", "energy_metric"])["value"]
    .sum()
    .reset_index()
)

df_plot = (
    df_yearly.pivot(index="year", columns="energy_metric", values="value")
    .reset_index()
    .sort_values("year")
)

# 2. Create Figure
fig_2030 = go.Figure()

available_low = [c for c in low_carbon_metrics if c in df_plot.columns]
for col in available_low:
    fig_2030.add_trace(go.Scatter(
        x=df_plot["year"],
        y=df_plot[col],
        name=col.replace("_electricity", "").title(),
        mode="lines",
        stackgroup="one",
        line=dict(width=0.5),
        fillcolor=color_map.get(col),
        visible=True
    ))

available_fossil = [c for c in fossil_metrics if c in df_plot.columns]
for col in available_fossil:
    fig_2030.add_trace(go.Scatter(
        x=df_plot["year"],
        y=df_plot[col],
        name=col.replace("_electricity", "").title(),
        mode="lines",
        stackgroup="one",
        line=dict(width=0.5),
        fillcolor=color_map.get(col),
        visible=False
    ))

# 3. Layout and Filter Menu
fig_2030.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Low-Carbon Mix",
                    method="update",
                    args=[
                        {"visible": [True]*len(available_low) + [False]*len(available_fossil)},
                        {"title": f"{title}: Low-Carbon Projection to 2030"}
                    ]
                ),
                dict(
                    label="Fossil Fuel Mix",
                    method="update",
                    args=[
                        {"visible": [False]*len(available_low) + [True]*len(available_fossil)},
                        {"title": f"{title}: Fossil Fuel Projection to 2030"}
                    ]
                )
            ],
            direction="down",
            x=0.01,
            xanchor="left",
            y=1.15,
            bgcolor="#FFF8E7",
            bordercolor="#383838",
            font=dict(size=14, color="#383838")
        )
    ],
    height=750,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    title=dict(
        text=f"{title}: Low-Carbon Projection to 2030",
        x=0.5,
        font=dict(size=24, color="#383838")
    ),
    xaxis=dict(
        title="Year",
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838")
    ),
    yaxis=dict(
        title="Electricity Generation (TWh)",
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838")
    ),
    font=dict(color="#383838"),
    margin=dict(t=120)
)

fig_2030.add_vline(
    x=2024,
    line_dash="dash",
    line_color="#383838",
    annotation_text="Prediction Start"
)

# Save
fig_2030.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\27_road_2030.html")
# Show
fig_2030.show()